# Stage A1 — Extract per-layer activations

**Experiment A — Representation Convergence (GPT-2 vs Pythia-160M)**

Hypothesis: *well-trained independent models share representations up to a
linear transformation.* The models were chosen from different labs,
architectures, tokenizers, and training corpora (OpenAI/WebText vs
EleutherAI/The Pile), both trained from scratch — so any alignment found
must have been discovered independently by each training run.


## What this stage does
Runs the **same 10,000 WikiText passages** through both models and saves the
mean-pooled hidden state of **every layer**. Also extracts a random-weights
copy of the Pythia architecture as a control baseline — it calibrates how
much similarity comes from architecture + input statistics alone, so the
trained-minus-random gap is the measured effect of *learning*.

**Why 10,000 and not 2,000.** Stage A3 fits a 768x768 linear map (~590K
parameters) per layer pair. With ~1,200 training rows that is 1.6 samples
per input dimension, and ridge regression in that regime does not become
noisy — it becomes **systematically pessimistic**, with no error raised.
The first run of this project failed its success criterion for exactly
this reason. 10,000 passages give ~10 samples/dim. The train split is used
because the validation split contains only ~1,646 passages over 100
characters; both splits are the same Wikipedia corpus, and nothing here is
trained, so no evaluation hygiene is affected (A3 holds out its own 25%).

## Expected output
`activations.npz` with `A_layers [L_A, N, d_A]`, `B_layers [L_B, N, d_B]`,
`R_layers` (random baseline). Runtime ~15 min on a T4.

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
os.environ["DATA_DIR"] = "/content/drive/MyDrive/convergence_experiment"

In [ ]:
# Install dependencies (once)
# !pip install torch transformers datasets numpy

In [ ]:
# Configuration and imports
import numpy as np
import os
from pathlib import Path

DATA_DIR = Path(os.environ.get("DATA_DIR", "."))
DATA_DIR.mkdir(parents=True, exist_ok=True)

import torch
from transformers import AutoModel, AutoTokenizer, AutoConfig

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
N_SAMPLES = 10000        # ~10 samples per input dimension for the A3 fit
MAX_LEN = 64
BATCH = 32
SEED = 0                 # seeds the random-weights control (reproducibility)

MODEL_A = "gpt2"
MODEL_B = "EleutherAI/pythia-160m"
print(f"device={DEVICE}  n_samples={N_SAMPLES}  seed={SEED}")

In [ ]:
def load_sentences(n, min_chars=100):
    """Return exactly `n` distinct WikiText passages of >= min_chars.

    Fixes vs original:
      * streams the corpus instead of materializing ds["text"] (the
        wikitext-103 train split is ~1.8M lines; the original built a
        Python list of all of them just to keep the first n)
      * de-duplicates, so repeated boilerplate lines do not inflate the
        effective sample size
      * FAILS LOUDLY if the corpus yields fewer than n passages -- this
        is the exact condition that caused the run-1 sample-starvation
        bias, and it must never again pass silently
      * reports which corpus was actually used
    """
    from datasets import load_dataset

    def stream(name, config):
        # streaming=True -> iterate lazily, stop as soon as we have n
        return load_dataset(name, config, split="train", streaming=True)

    for config in ("wikitext-103-raw-v1", "wikitext-2-raw-v1"):
        try:
            ds = stream("Salesforce/wikitext", config)
            seen, out = set(), []
            for row in ds:                      # lazy: no full download
                t = row["text"].strip()
                if len(t) < min_chars or t.startswith("="):   # skip headings
                    continue
                if t in seen:                   # skip exact duplicates
                    continue
                seen.add(t)
                out.append(t)
                if len(out) >= n:
                    break
            if len(out) >= n:
                print(f"corpus: {config} -> {len(out)} passages")
                return out
            print(f"corpus: {config} yielded only {len(out)} - trying next")
        except Exception as e:
            print(f"corpus {config} unavailable ({type(e).__name__}) - "
                  f"trying next")

    raise RuntimeError(
        f"Could not obtain {n} passages. Do NOT proceed with fewer: "
        f"the stitching fit needs ~10 samples per input dimension "
        f"(768 dims -> ~7,700 training rows). Fewer samples produce "
        f"systematically pessimistic R2, not noisier R2."
    )


@torch.no_grad()                       # no gradients: we only read activations
def extract(model, tokenizer, sentences):
    """Return [n_layers, n_samples, hidden_dim] -- mean-pooled per layer.

    Fixes vs original:
      * accumulates batches in a list and concatenates ONCE at the end
        (the original np.concatenate inside the loop is O(n^2) copying:
        every batch rewrote the whole array)
      * asserts the token mask is non-empty before dividing
    """
    model.eval().to(DEVICE)            # eval mode: disable dropout etc.
    chunks = []                        # list of [L, B, D] arrays

    for i in range(0, len(sentences), BATCH):
        batch = sentences[i:i + BATCH]
        # tokenize: pad to longest in batch, truncate long passages.
        # right padding (the default for these tokenizers) is correct for
        # decoder-only models: causal attention means real tokens never
        # attend to trailing pads.
        enc = tokenizer(batch, return_tensors="pt", padding=True,
                        truncation=True, max_length=MAX_LEN).to(DEVICE)

        # output_hidden_states=True -> tuple of [B, T, D], one entry per
        # layer INCLUDING the embedding output (so 13 for a 12-block model)
        out = model(**enc, output_hidden_states=True)

        mask = enc["attention_mask"].unsqueeze(-1)      # [B, T, 1]
        denom = mask.sum(1)                             # [B, 1] real tokens
        assert (denom > 0).all(), "empty sequence in batch"

        # mean-pool over real tokens only: sum(h * mask) / count.
        # pooling is what makes the two models comparable at all -- their
        # tokenizations differ, so there is no token-level correspondence,
        # but there IS a sentence-level one.
        pooled = [((h * mask).sum(1) / denom).float().cpu().numpy()
                  for h in out.hidden_states]
        chunks.append(np.stack(pooled))                 # [L, B, D]

        if (i // BATCH) % 10 == 0:
            print(f"  {i}/{len(sentences)}")

    return np.concatenate(chunks, axis=1)               # [L, N, D]


def main():
    # --- reproducibility: the random-weights control MUST be seeded,
    # --- otherwise the baseline differs between runs and the
    # --- trained-minus-random gap is not exactly reproducible.
    torch.manual_seed(SEED)
    np.random.seed(SEED)

    print("Loading sentences...")
    sentences = load_sentences(N_SAMPLES)
    print(f"{len(sentences)} sentences")
    # explicit power check, printed with the data it refers to
    print(f"samples/dim = {len(sentences) * 0.75 / 768:.1f} "
          f"(>= ~5 recommended for the 768x768 stitching fit)")

    def build(name, tok_source=None):
        """Load tokenizer + model, run extraction, free GPU memory."""
        tok = AutoTokenizer.from_pretrained(tok_source or name)
        if tok.pad_token is None:
            # GPT-2/NeoX have no pad token; reuse EOS. Harmless because
            # the attention mask excludes pads from pooling anyway.
            tok.pad_token = tok.eos_token
        model = AutoModel.from_pretrained(name)
        acts = extract(model, tok, sentences)
        del model
        torch.cuda.empty_cache()
        return acts, tok

    print(f"\nExtracting from {MODEL_A}...")
    A, _ = build(MODEL_A)

    print(f"\nExtracting from {MODEL_B}...")
    B, tok_b = build(MODEL_B)

    print("\nExtracting random-weights baseline (untrained Pythia arch)...")
    # from_config (not from_pretrained) -> same architecture, fresh random
    # weights, zero training steps: the control that calibrates how much
    # similarity comes from architecture + input statistics alone.
    cfg = AutoConfig.from_pretrained(MODEL_B)
    rand_model = AutoModel.from_config(cfg)
    R = extract(rand_model, tok_b, sentences)   # same tokenizer as Pythia
    del rand_model
    torch.cuda.empty_cache()                    # was missing in the original

    # --- shape validation: A2/A3 assume identical layer counts and row
    # --- alignment (row i of every array = sentence i). Catch mismatches
    # --- here rather than as a confusing result three notebooks later.
    assert A.shape[1] == B.shape[1] == R.shape[1] == len(sentences), \
        f"row mismatch: {A.shape}, {B.shape}, {R.shape}"
    assert B.shape[0] == R.shape[0], "control must match Pythia layer count"
    if A.shape[0] != B.shape[0]:
        print(f"NOTE: layer counts differ ({A.shape[0]} vs {B.shape[0]}); "
              f"A2/A3 compare the {min(A.shape[0], B.shape[0])} common layers")

    out = DATA_DIR / "activations.npz"
    np.savez_compressed(str(out), A_layers=A, B_layers=B, R_layers=R,
                        n_samples=len(sentences), seed=SEED)
    print(f"\nSaved: A {A.shape}, B {B.shape}, R {R.shape} -> {out}")

## Review notes — issues found and fixed in this version

| # | Issue | Why it mattered | Fix |
|---|---|---|---|
| 1 | `sents[:n]` returned fewer than `n` silently | reproduces the run-1 sample-starvation bias undetectably | raise `RuntimeError`; print samples/dim |
| 2 | random control not seeded | the baseline every result is measured against changed per run | `torch.manual_seed(SEED)`, seed saved in the npz |
| 3 | `np.concatenate` inside the batch loop | O(n^2) copying of the whole array each batch | accumulate in a list, concatenate once |
| 4 | `ds["text"]` materialized the full column | ~1.8M strings loaded to keep 10,000 | `streaming=True` + early break |
| 5 | `rand_model` never freed | stayed resident on GPU through the save | `del` + `empty_cache()` |
| 6 | no shape validation | A2/A3 assume row *i* = sentence *i* and matching layer counts | asserts at the source |
| 7 | silent corpus fallback | results would not record which corpus was used | prints the corpus actually used |

Also added: exact-duplicate passages dropped (repeated boilerplate inflates
the nominal sample count without adding information), and WikiText heading
lines (`= Title =`) skipped since they are not prose.

**Deliberately unchanged:** right-padding. These are decoder-only models
with causal attention, so real tokens never attend to trailing pads, and
mean-pooling excludes pads via the mask. Left-padding would be the risky
choice here.

In [ ]:
# Run the extraction
main()